In [1]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

In [3]:
from pathlib import Path

CANDIDATES = [
    Path("../data/processed/price_review_task4"),
    Path("data/processed/price_review_task4"),
    Path("."),
    Path(".."),
]

DATA_DIR = next(
    (
        p / "price_review_task4"
        if (p / "price_review_task4").is_dir()
        else p
        for p in CANDIDATES
        if (
            (p / "price_review_task4").is_dir()
            or (
                p.name == "price_review_task4"
                and p.is_dir()
            )
        )
    ),
    None
)



print("DATA_DIR:", DATA_DIR.resolve())
print("Files:", [f.name for f in DATA_DIR.iterdir()])

DATA_DIR: E:\DEPI-Grad\DEPI-GRADUATION-PROJECT\data\processed\price_review_task4
Files: ['review_sheet.csv', 'review_sheet_filled (2).csv', 'train_scored_4_1.csv', 'val_scored_4_1.csv']


In [4]:
review_Sheet = pd.read_csv(DATA_DIR / "review_sheet_filled (2).csv")

print("review sheet :", review_Sheet.shape)

review sheet : (100, 17)


In [13]:
review_Sheet.head(10)

,ad_id,ad_url,side,title,brand,model_key,price,ref_median_price,price_ratio,mad_z,price_review_reason,price_context,decision,fixed_price,fixed_model_key,error_type,notes,reason_count
0,208989958,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 17 برو ماكس,apple,apple iphone 17 pro max,1000.0,83000.0,0.01,-29.81,price_placeholder;ratio_low;mad_low,NaN,drop,NaN,NaN,placeholder_price,"Flagged by 3 checks (price_placeholder, ratio_...",3
1,208954191,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 17 برو ماكس,apple,apple iphone 17 pro max,1000.0,83000.0,0.01,-29.81,price_placeholder;ratio_low;mad_low,NaN,drop,NaN,NaN,placeholder_price,"Flagged by 3 checks (price_placeholder, ratio_...",3
2,208983975,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 17 برو,apple,apple iphone 17 pro,1000.0,74000.0,0.01,-20.95,price_placeholder;ratio_low;mad_low,NaN,drop,NaN,NaN,placeholder_price,"Flagged by 3 checks (price_placeholder, ratio_...",3
3,209009851,/ad/%D8%B3%D8%A7%D9%85%D8%B3%D9%88%D9%86%D8%AC...,lowest,سامسونج جلاكسي اس 26 الترا,samsung,samsung s26 ultra,1000.0,67000.0,0.01,-28.36,price_placeholder;ratio_low;mad_low,NaN,drop,NaN,NaN,placeholder_price,"Flagged by 3 checks (price_placeholder, ratio_...",3
4,208942594,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 17 برو ماكس,apple,apple iphone 17 pro max,1500.0,83000.0,0.02,-27.07,ratio_low;mad_low,NaN,unresolved,NaN,NaN,unresolved,price = 2% of ref; not a real phone price. Cau...,2
5,208984516,/ad/%D8%B9%D8%B1%D8%B1%D8%B1%D8%B6-%D8%AE%D8%B...,lowest,عرررض خطيررر ع الايفونن 17 برماكس *IPHoooONE ...,apple,apple iphone 17 pro max,2000.0,83000.0,0.02,-25.13,ratio_low;mad_low,NaN,unresolved,NaN,NaN,unresolved,Spam/obfuscated title + price ~2-3% of ref: li...,2
6,208986825,/ad/%D8%A7%D9%8A%D9%88%D8%A7%D8%A7%D8%A7%D8%A7...,lowest,ايواااا الجديد شديد ايفون١٧بروووووماكس *iPhon...,apple,apple iphone 17 pro max,2500.0,83000.0,0.03,-23.62,ratio_low;mad_low,NaN,unresolved,NaN,NaN,unresolved,Spam/obfuscated title + price ~2-3% of ref: li...,2
7,209005186,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 16 برو ماكس,apple,apple iphone 16 pro max,2000.0,63000.0,0.03,-23.27,ratio_low;mad_low,NaN,unresolved,NaN,NaN,unresolved,price = 3% of ref; not a real phone price. Cau...,2
8,208976461,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 17 برو ماكس ميني كوبي,apple,apple iphone 17 pro max,3000.0,83000.0,0.04,-22.39,ratio_low;mad_low,NaN,drop,NaN,NaN,replica_clone,"Title says high/first copy (replica), not a ge...",2
9,208966198,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 17 برو ماكس,apple,apple iphone 17 pro max,3000.0,83000.0,0.04,-22.39,ratio_low;mad_low,NaN,unresolved,NaN,NaN,unresolved,price = 4% of ref; not a real phone price. Cau...,2


In [17]:
review_Sheet = review_Sheet.copy()

review_Sheet["reason_count"] = (
    review_Sheet["price_review_reason"]
    .fillna("")
    .apply(lambda x: len([r.strip() for r in x.split(";") if r.strip()]))
)

In [18]:
mask = review_Sheet["reason_count"] >= 2

review_Sheet.loc[mask, "decision"] = "drop"

review_Sheet.loc[mask, "notes"] = (
    "Dropped based on the cleaning rule: "
    "more than two price review reasons."
)

In [19]:
display(
    review_Sheet[
        [
            "title",
            "price",
            "price_review_reason",
            "reason_count",
            "decision",
            "notes"
        ]
    ]
)

,title,price,price_review_reason,reason_count,decision,notes
0,آبل - آيفون 17 برو ماكس,1000.0,price_placeholder;ratio_low;mad_low,3,drop,Dropped based on the cleaning rule: more than ...
1,آبل - آيفون 17 برو ماكس,1000.0,price_placeholder;ratio_low;mad_low,3,drop,Dropped based on the cleaning rule: more than ...
2,آبل - آيفون 17 برو,1000.0,price_placeholder;ratio_low;mad_low,3,drop,Dropped based on the cleaning rule: more than ...
3,سامسونج جلاكسي اس 26 الترا,1000.0,price_placeholder;ratio_low;mad_low,3,drop,Dropped based on the cleaning rule: more than ...
4,آبل - آيفون 17 برو ماكس,1500.0,ratio_low;mad_low,2,drop,Dropped based on the cleaning rule: more than ...
...,...,...,...,...,...,...
95,اوبو ايه 54,6500.0,NaN,0,unresolved,price = 1.74x ref; check description (variant/...
96,اوبو رينو 6,9500.0,NaN,0,unresolved,price = 1.73x ref; check description (variant/...
97,Xiaomi 15,19000.0,NaN,0,unresolved,Ref median 11000 looks low for Xiaomi 15: key ...
98,آيفون 14 برو ماكس مساحة سوداء 1 تيرابايت,68000.0,mad_high,1,unresolved,Title says 1TB: high price likely a storage va...


In [20]:
print(review_Sheet["decision"].value_counts(dropna=False))

decision
drop          56
unresolved    43
fix            1
Name: count, dtype: int64


Dropped 56 
kept 43 
fixed 1

In [ ]:
review_Sheet.to_excel("review_Sheet_final_100_rows.xlsx", index=False)